In [9]:
# Imports & The Triton Patch

import os
import sys
import torch

# PATCH 1: Disable hf_transfer which causes silent download hangs on Windows
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0" 

# PATCH 2: Fix Triton 3.x AttrsDescriptor crash on Windows
import triton
class AttrsDescriptor: pass
if hasattr(triton, 'compiler') and hasattr(triton.compiler, 'compiler'):
    triton.compiler.compiler.AttrsDescriptor = AttrsDescriptor
if hasattr(triton, 'backends') and hasattr(triton.backends, 'compiler'):
    triton.backends.compiler.AttrsDescriptor = AttrsDescriptor

from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

print("✅ Environment patched and imported successfully.")

✅ Environment patched and imported successfully.


In [10]:
#Load, Shuffle, Split & Inspect Data

from datasets import load_dataset
import os
import json

DATASET_PATH = "./data/complete_dataset.jsonl" 
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"❌ ERROR: Could not find '{DATASET_PATH}'.")

print(f"✅ Found dataset at {DATASET_PATH}. Loading...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print("Shuffling dataset...")
dataset = dataset.shuffle(seed=42)

# --- NEW POC LOGIC: Slice to 10% ---
subset_size = int(len(dataset) * 0.10) 
dataset = dataset.select(range(subset_size))
print(f"✅ Reduced dataset to {len(dataset)} samples for POC speed-run.")
# -----------------------------------

split_dataset = dataset.train_test_split(test_size=0.10, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"✅ Train set size: {len(train_dataset)}")
print(f"✅ Validation set size: {len(val_dataset)}")

print("\n--- [INSPECT] Raw Sample from Training Data ---")
print(json.dumps(train_dataset[0], indent=2, ensure_ascii=False))

✅ Found dataset at ./data/complete_dataset.jsonl. Loading...
Shuffling dataset...
✅ Reduced dataset to 948 samples for POC speed-run.
✅ Train set size: 853
✅ Validation set size: 95

--- [INSPECT] Raw Sample from Training Data ---
{
  "messages": [
    {
      "role": "system",
      "content": "You are an expert intent extraction assistant. Analyze the conversation and output ONLY valid JSON."
    },
    {
      "role": "user",
      "content": "USER: Documents required for Gold loan?\nBOT: To apply for a gold loan, you'll need to submit any one of the following documents:<br><br>- Aadhaar card<br>- Voter ID card<br>- Passport<br>- Driving licence<br>- NREGA job card<br>- Letter issued by National Population Registration<br><br>PAN card is not required, but if you're applying for a gold loan of Rs. 5 lakh or above, you'll need to submit your PAN card. Ready to unlock the value of your gold? 🎉\nUSER: [PAN_REDACTED]\nBOT: Here are the fantastic benefits of a gold loan with Bajaj Finance

In [11]:
# Downlaod the model manually using huggingface_hub to avoid Unsloth's buggy downloader on Windows

from huggingface_hub import snapshot_download
import os

# Define local path
LOCAL_MODEL_PATH = "./models/sarvam-1"
os.makedirs(LOCAL_MODEL_PATH, exist_ok=True)

print("Downloading sarvam-1 manually (This bypasses Unsloth's buggy Windows downloader)...")
# This will show reliable progress bars and can resume if your internet drops
snapshot_download(
    repo_id="sarvamai/sarvam-1",
    local_dir=LOCAL_MODEL_PATH,
    local_dir_use_symlinks=False,
    ignore_patterns=["*.pt", "*.bin", "*.msgpack", "*.h5"] # Only download the modern safetensors format
)

print(f"✅ Model files downloaded successfully to {LOCAL_MODEL_PATH}")

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 346.60it/s]

✅ Model files downloaded successfully to ./models/sarvam-1


In [12]:
from unsloth import FastLanguageModel
import torch

# Load from local folder
LOCAL_MODEL_PATH = "./models/sarvam-1"
MAX_SEQ_LENGTH = 1024 

print(f"Loading model from local directory: {LOCAL_MODEL_PATH}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LOCAL_MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, 
    load_in_4bit=True, 
)

print(f"✅ Model loaded successfully.")
print(f"Current GPU VRAM Used: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB")

# Formatting function for SFTTrainer
def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

print("\nFormatting datasets with chat template...")
# FIX: Removed num_proc completely to force execution in the main Jupyter process
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print("\n--- [INSPECT] Formatted Text Sample (First 500 chars) ---")
print(train_dataset[0]['text'][:500] + "...")

Loading model from local directory: ./models/sarvam-1...
==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5050 Laptop GPU. Num GPUs = 1. Max memory: 7.96 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading weights: 100%|██████████| 255/255 [00:03<00:00, 69.21it/s]
Unsloth: Will load ./models/sarvam-1 as a legacy tokenizer.


Unsloth: ./models/sarvam-1 has no pad_token. Using pad_token = <unk>.
✅ Model loaded successfully.
Current GPU VRAM Used: 3498.97 MB

Formatting datasets with chat template...


Map: 100%|██████████| 95/95 [00:00<00:00, 8093.49 examples/s]


--- [INSPECT] Formatted Text Sample (First 500 chars) ---
<s>[INST] <<SYS>>
You are an expert intent extraction assistant. Analyze the conversation and output ONLY valid JSON.
<</SYS>>

USER: Documents required for Gold loan?
BOT: To apply for a gold loan, you'll need to submit any one of the following documents:<br><br>- Aadhaar card<br>- Voter ID card<br>- Passport<br>- Driving licence<br>- NREGA job card<br>- Letter issued by National Population Registration<br><br>PAN card is not required, but if you're applying for a gold loan of Rs. 5 lakh or abo...


In [13]:
# Applying LoRA Adapters

print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16, 
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj", 
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32, 
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth", # Critical for Laptop GPUs to prevent OOM
    random_state=42
)

print("\n[INSPECT] Trainable Parameters:")
model.print_trainable_parameters()

Applying LoRA adapters...

[INSPECT] Trainable Parameters:
trainable params: 23,969,792 || all params: 2,549,057,536 || trainable%: 0.9403


In [14]:
# Training with SFTTrainer

from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
import os

OUTPUT_DIR = "./outputs/sarvam1-intent-finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=1, # CRITICAL: Must be 1 to avoid Windows multiprocessing tokenizer errors
    packing=False, 
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4, # Effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none", 
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting training... (Monitor the progress bar below)")
trainer_stats = trainer.train()

Initializing SFTTrainer...
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]: 100%|██████████| 853/853 [00:00<00:00, 1763.03 examples/s]


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"]: 100%|██████████| 95/95 [00:00<00:00, 1641.07 examples/s]


Starting training... (Monitor the progress bar below)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 853 | Num Epochs = 3 | Total steps = 321
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 23,969,792 of 2,549,057,536 (0.94% trained)


Step,Training Loss,Validation Loss
50,0.799128,0.683813
100,0.581288,0.459175
150,0.398047,0.390220
200,0.386371,0.345818
250,0.332158,0.334094
300,0.330936,0.321692
321,0.324911,0.321233


Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./outputs/sarvam1-intent-finetuned\checkpoint-50.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./outputs/sarvam1-intent-finetuned\checkpoint-100.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./outputs/sarvam1-intent-finetuned\checkpoint-150.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./outputs/sarvam1-intent-finetuned\checkpoint-200.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./outputs/sarvam1-intent-finetuned\checkpoint-250.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/sarvam1-intent-finetuned\checkpoint-321\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./outputs/sarvam1-intent-finetuned\checkpoint-321.


In [15]:
# Saving the LoRA Adapters

import os

OUTPUT_DIR = "./outputs/sarvam1-intent-finetuned"
final_model_path = os.path.join(OUTPUT_DIR, "final_lora_model")

print(f"Saving LoRA adapters to {final_model_path}...")
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("✅ Adapters saved successfully! You can now safely restart the kernel without losing your work.")

Saving LoRA adapters to ./outputs/sarvam1-intent-finetuned\final_lora_model...


Unsloth: Restored added_tokens_decoder metadata in ./outputs/sarvam1-intent-finetuned\final_lora_model\tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./outputs/sarvam1-intent-finetuned\final_lora_model.


✅ Adapters saved successfully! You can now safely restart the kernel without losing your work.


In [14]:
import os
import json
import torch
from pydantic import BaseModel, Field
from typing import List, Literal # <-- Added Literal

# --- MONKEY PATCH: Fix for transformers 4.45+ and lm-format-enforcer ---
import transformers
try:
    from transformers.tokenization_utils_base import PreTrainedTokenizerBase
    transformers.tokenization_utils.PreTrainedTokenizerBase = PreTrainedTokenizerBase
except ImportError:
    pass
# -----------------------------------------------------------------------

# 1. Define the UPDATED Strict Production Schema
class SearchQuery(BaseModel):
    product: str
    topic: str
    # STRICT ENFORCEMENT: Only these 3 words are now mathematically possible
    confidence: Literal["low", "medium", "high"] 
    ambiguousProducts: List[str] = Field(default_factory=list)
    keywordSearchText: str

class IntentOutput(BaseModel):
    searchQueries: List[SearchQuery] = Field(default_factory=list)
    noRetrievalNeeded: bool

# 2. Load Model and Apply Unsloth Inference Patch
from unsloth import FastLanguageModel
from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn

LOCAL_MODEL_PATH = "./models/sarvam-1"
LORA_PATH = "./outputs/sarvam1-intent-finetuned/final_lora_model"

print("Loading Base Model + LoRA for Inference...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

# CRITICAL: Unsloth requires this to re-enable KV caching and switch out of training mode
model = FastLanguageModel.for_inference(model) 

# 3. Setup the Grammar Enforcer with the NEW Schema
parser = JsonSchemaParser(IntentOutput.model_json_schema())
prefix_allowed_tokens_fn = build_transformers_prefix_allowed_tokens_fn(tokenizer, parser)

print("✅ Model loaded and STRICT Grammar Enforcer initialized.")

Loading Base Model + LoRA for Inference...
==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5050 Laptop GPU. Num GPUs = 1. Max memory: 7.96 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 255/255 [00:03<00:00, 80.50it/s]
Unsloth: Will load ./outputs/sarvam1-intent-finetuned/final_lora_model as a legacy tokenizer.


✅ Model loaded and STRICT Grammar Enforcer initialized.


In [19]:
test_conversation_retrieval_flow = """
[{\"role\":\"USER\",\"message\":\"\"},{\"role\":\"BOT\",\"message\":\"Congratulations <b style=\\\"font-size:16px; color:blue;\\\">Akshay Kumar</b>, You have a special pre-approved loan offer of upto <b style=\\\"font-size:16px; color:blue;\\\">₹140000</b>. Please confirm your details to proceed and generate your sanction letter.\"},{\"role\":\"USER\",\"message\":\"Confirm\"},{\"role\":\"BOT\",\"message\":\"Thank you for confirming your details.</br><b>Enter your birth date in the required DD/MM/YYYY format.\"},{\"role\":\"USER\",\"message\":\"15/06/2004\"},{\"role\":\"BOT\",\"message\":\"Thanks! Just few more details!</br><b>Type your PAN as it appears on your card.\"},{\"role\":\"USER\",\"message\":\"No pan card\"},{\"role\":\"BOT\",\"message\":\"Amhi tumcha concern samajto. Pan, tumcha PAN number dene tumchya application sathi move forward karanyasathi avashyak aahe.</br><b>Kripaya tumcha official PAN enter kara.\"}]
"""

In [20]:
from json_repair import repair_json

# UPDATED SYSTEM PROMPT (Business Rules)
BAJAJ_FINANCE_SYSTEM_PROMPT = """You are an expert intent extraction AI for Bajaj Finance. Analyze the multi-turn customer-bot conversation and output a strict JSON object. 
A 'product' refers strictly to financial services offered by Bajaj Finance (e.g., Personal Loan, Business Loan, Secured Loan, Two-Wheeler (TW) Loan, Insta EMI Card, Home Loan, Gold Loan, etc.). 
If the user is asking for information about, applying for, or showing active intent/progress towards acquiring any of these financial services, extract search queries and set 'noRetrievalNeeded' to false. 
If the user is engaging in chit-chat, asking general non-financial questions, or resolving issues with existing accounts where no new service intent is shown, set 'noRetrievalNeeded' to true. 

CRITICAL RULES FOR FIELDS:
- 'confidence': Must be exactly one of 'low', 'medium', or 'high'. (High = explicit product intent, Medium = ambiguous, Low = weak intent).
- 'keywordSearchText': Extract clean alphanumeric search keywords. DO NOT include currency symbols (like ₹, $) or punctuation.
Do not include any explanations, markdown formatting, or code block ticks. Output ONLY raw JSON matching the schema."""

# Use the long Insta EMI Card conversation
messages = [
    {"role": "system", "content": BAJAJ_FINANCE_SYSTEM_PROMPT},
    {"role": "user", "content": test_conversation_retrieval_flow.strip()}
]
print("test_conversation_retrieval_flow:", test_conversation_retrieval_flow.strip())
inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=True, 
    add_generation_prompt=True, 
    return_tensors="pt"
).to("cuda")

print("Generating with Strict Categorical Constraints...")
outputs = model.generate(
    inputs, 
    max_new_tokens=512, 
    temperature=0.1, 
    do_sample=True,
    prefix_allowed_tokens_fn=prefix_allowed_tokens_fn, 
    pad_token_id=tokenizer.eos_token_id
)

# 1. Decode the raw text
generated_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print("\n--- [DEBUG] Raw Model Output ---")
print(repr(generated_text))
print("-------------------------------\n")

# 2. BULLETPROOF HEALING (Production Standard)
parsed_json = repair_json(generated_text, return_objects=True)

print("✅ Final Production-Ready JSON:")
print(json.dumps(parsed_json, indent=2, ensure_ascii=False))

test_conversation_retrieval_flow: [{"role":"USER","message":""},{"role":"BOT","message":"Congratulations <b style=\"font-size:16px; color:blue;\">Akshay Kumar</b>, You have a special pre-approved loan offer of upto <b style=\"font-size:16px; color:blue;\">₹140000</b>. Please confirm your details to proceed and generate your sanction letter."},{"role":"USER","message":"Confirm"},{"role":"BOT","message":"Thank you for confirming your details.</br><b>Enter your birth date in the required DD/MM/YYYY format."},{"role":"USER","message":"15/06/2004"},{"role":"BOT","message":"Thanks! Just few more details!</br><b>Type your PAN as it appears on your card."},{"role":"USER","message":"No pan card"},{"role":"BOT","message":"Amhi tumcha concern samajto. Pan, tumcha PAN number dene tumchya application sathi move forward karanyasathi avashyak aahe.</br><b>Kripaya tumcha official PAN enter kara."}]


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating with Strict Categorical Constraints...

--- [DEBUG] Raw Model Output ---
'{\n  "searchQueries": [],\n  "noRetrievalNeeded": true\n}'
-------------------------------

✅ Final Production-Ready JSON:
{
  "searchQueries": [],
  "noRetrievalNeeded": true
}
